# NEXUS LLM Extension: Mock-Training Demo

Runs  with no GPU, no `mujoco_playground`,
and no LLM API access required. Generation uses the deterministic mock
backend, and "training" uses `nexus_continuous.llm.mock_training.MockTrainer`.

Steps:
1. Bootstrap `jax` (real if installed, else
   `_jax_stub/` for `interpreter.py`'s rule/reward compilation).
2. Generate a skillset with the mock LLM backend (seeded, deterministic).
3. Compile it with `interpreter.py` into a runnable policy module and sanity-check it.
4. Multi-seed hand-written vs. LLM comparison via `mock_training.py` and plot.
5. Interactive refinement loop across iterations and seeds (+ plotting).

In [1]:
import sys, os

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from nexus_continuous.llm.jax_bootstrap import ensure_jax
used_stub = ensure_jax()
print("Using bundled jax stub:", used_stub)

Using bundled jax stub: False


## 1. Generate a skillset with the mock backend

In [2]:
from nexus_continuous.llm.client import LLMClient, LLMConfig, MockSkillGenerator
from nexus_continuous.llm.pipeline import generate_skillset

env_name = "CartpoleBalance"
fields = ("cart_position", "pole_angle", "cart_velocity", "pole_angular_velocity")
task_description = "Keep the pole upright and centered while minimizing oscillations."

seed = 0
client = LLMClient(
    LLMConfig(backend="mock", seed=seed), 
    mock_generator=MockSkillGenerator(fields, seed=seed)
)
skillset = generate_skillset(
    env_name=env_name,
    observation_schema="\n".join(fields),
    task_description=task_description,
    client=client,
    allowed_fields=set(fields),
)
for s in skillset.skills:
    print(f"- {s.name}: activation_rule={s.activation_rule!r}, {len(s.reward_terms)} reward term(s)")

- skill_0_cart_position: activation_rule='abs(cart_position) > 0.15', 2 reward term(s)
- skill_1_pole_angle: activation_rule='abs(pole_angle) > 0.25', 2 reward term(s)
- skill_2_cart_velocity: activation_rule='abs(cart_velocity) > 0.35', 2 reward term(s)


## 2. Compile with `interpreter.py` and sanity-check

In [3]:
from dataclasses import asdict
import jax.numpy as jnp
from nexus_continuous.llm.interpreter import make_policy_module

policy_module = make_policy_module(asdict(skillset), field_names=fields)
print("Skills:", policy_module.SKILL_NAMES)

rng = np.random.default_rng(0)
obs = jnp.asarray(rng.standard_normal((8, len(fields))).astype("float32"))
action = jnp.asarray(rng.standard_normal((8, 1)).astype("float32"))
done = jnp.zeros((8,), dtype=bool)

rewards = policy_module.skill_rewards(obs, obs, action, None, done, None)
mask = policy_module.skill_mask(obs, None)
meta_policy = policy_module.symbolic_meta_policy(obs, None)

print("skill_rewards shape:", rewards.shape)
print("skill_mask shape:", mask.shape)
print("symbolic_meta_policy:", np.asarray(meta_policy))
print(policy_module.explain_policy())

Skills: ('skill_0_cart_position', 'skill_1_pole_angle', 'skill_2_cart_velocity')
skill_rewards shape: (8, 3)
skill_mask shape: (8, 3)
symbolic_meta_policy: [2 0 0 0 0 1 0 0]
LLM-generated skills: skill_0_cart_position, skill_1_pole_angle, skill_2_cart_velocity


## 3. Multi-seed hand-written vs LLM comparison

In [4]:
from nexus_continuous.llm.mock_training import mock_train_fn, hand_written_baseline_metrics

n_seeds = 5
hand_runs = [hand_written_baseline_metrics(env_name, seed=s) for s in range(n_seeds)]
llm_runs = [mock_train_fn(asdict(skillset), seed=s) for s in range(n_seeds)]

def summarize(runs):
    vals = [r["returns/env_reward_mean"] for r in runs]
    return {"mean": float(np.mean(vals)), "std": float(np.std(vals))}

hand_summary = summarize(hand_runs)
llm_summary = summarize(llm_runs)
print("Handwritten:", hand_summary)
print("LLM:        ", llm_summary)

Handwritten: {'mean': 8.119942036207323, 'std': 0.42996328236110337}
LLM:         {'mean': 8.440326381558796, 'std': 0.5469903718768575}


In [5]:
from nexus_continuous.llm.plot import plot_comparison

os.makedirs("plots", exist_ok=True)
path = plot_comparison(hand_summary, llm_summary, env_name, "plots/comparison.png")
plt.figure(figsize=(4.5, 4))
plt.imshow(plt.imread(path))
plt.axis("off")
plt.show()

C:\Users\User\AppData\Local\Temp\ipykernel_9924\3387026508.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Interactive refinement loop

In [6]:
from nexus_continuous.llm.pipeline import LLMSkillPipeline
from nexus_continuous.llm.refinement_loop import LLMRefinementLoop, RefinementConfig
from nexus_continuous.llm.mock_training import MockTrainer

def run_refinement(seed, iterations=6):
    client = LLMClient(
        LLMConfig(backend="mock", seed=seed), 
        mock_generator=MockSkillGenerator(fields, seed=seed)
    )
    pipeline = LLMSkillPipeline(client)
    loop = LLMRefinementLoop(pipeline, client)
    cfg = RefinementConfig(
        env_name=env_name,
        observation_schema="\n".join(fields),
        task_description=task_description,
        num_iterations=iterations,
        allowed_fields=set(fields),
    )
    return loop.run(cfg, MockTrainer(seed=seed))

results = {seed: run_refinement(seed).history for seed in range(4)}
for seed, history in results.items():
    vals = [rec.metrics["returns/env_reward_mean"] for rec in history]
    print(f"seed {seed}: env_reward_mean per iteration = {[round(v, 2) for v in vals]}")


[LLM LOOP] iteration 0
Metrics:
returns/env_reward_mean: 9.022912837787436
returns/skill_reward_mean: 4.58793422535328
policy_diag/primary_success_rate: 0.9203471597049425
policy_diag/primary_goal_metric: 0.8873399769095793
env/returned_episode_returns: 9.022912837787436

[LLM LOOP] iteration 1
Metrics:
returns/env_reward_mean: 9.3921035278641
returns/skill_reward_mean: 4.681663678165443
policy_diag/primary_success_rate: 0.82661883182422
policy_diag/primary_goal_metric: 0.860406576383707
env/returned_episode_returns: 9.3921035278641

[LLM LOOP] iteration 2
Metrics:
returns/env_reward_mean: 9.636928860988276
returns/skill_reward_mean: 4.495639697894402
policy_diag/primary_success_rate: 0.8225013891279576
policy_diag/primary_goal_metric: 0.9486887343262882
env/returned_episode_returns: 9.636928860988276

[LLM LOOP] iteration 3
Metrics:
returns/env_reward_mean: 10.196830243043664
returns/skill_reward_mean: 5.077542398255073
policy_diag/primary_success_rate: 0.9288645295115064
policy_diag

In [7]:
from nexus_continuous.llm.plot import plot_refinement, plot_multi_seed_refinement

plot_refinement(results[0], "plots/refinement_seed0.png", title=f"{env_name} refinement (seed 0)")
path = plot_multi_seed_refinement(results, "plots/refinement_all_seeds.png", title=f"{env_name} refinement across seeds")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].imshow(plt.imread("plots/refinement_seed0.png")); axes[0].axis("off")
axes[1].imshow(plt.imread(path)); axes[1].axis("off")
plt.tight_layout()
plt.show()

C:\Users\User\AppData\Local\Temp\ipykernel_9924\3599648891.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
